# Semantic Views & the AI-BI Stack on Snowflake

A hands-on tour of one governed metric layer feeding four consumers. Scenario: an SMS/MMS
marketing platform for e-commerce brands — shoppers opt in by keyword, brands send broadcasts
and automated flows, and store orders attribute back to the send that drove them.

### What you'll do
1. Explore the star-schema data
2. Inspect and query the **semantic view** — the single source of truth
3. See the **three ways** to create the view (DDL, YAML/dbt, no-code wizard)
4. Run **Cortex Analyst**-style questions as governed SQL
5. Run **Cortex Search** over the document corpus
6. Blend both with the **Cortex Agent**
7. Recap the positioning and export the view to YAML for Git

**Prerequisite:** run `lab/setup.sql` first. It builds all data + objects and is idempotent.

---
## Section 1: Connect & explore the data

Everything runs in the current Snowflake session (`get_active_session()`) — no local auth.

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE DATABASE SMS_MARKETING_DEMO").collect()
session.sql("USE SCHEMA CORE").collect()
session.sql("USE WAREHOUSE SMS_MARKETING_WH").collect()
print("Connected to SMS_MARKETING_DEMO.CORE")

In [ ]:
# Star schema at a glance: row counts per table
session.sql("""
    SELECT 'DIM_BRAND' AS table_name, COUNT(*) AS rows FROM DIM_BRAND
    UNION ALL SELECT 'DIM_SUBSCRIBER', COUNT(*) FROM DIM_SUBSCRIBER
    UNION ALL SELECT 'DIM_CAMPAIGN', COUNT(*) FROM DIM_CAMPAIGN
    UNION ALL SELECT 'FACT_MESSAGE', COUNT(*) FROM FACT_MESSAGE
    UNION ALL SELECT 'FACT_ORDER', COUNT(*) FROM FACT_ORDER
    ORDER BY rows DESC
""").show()

In [ ]:
# Peek at the grain: one message row, one order row
session.sql("SELECT * FROM FACT_MESSAGE LIMIT 5").show()
session.sql("SELECT * FROM FACT_ORDER LIMIT 5").show()

---
## Section 2: The semantic view — the single source of truth

`SMS_MARKETING_SV` is a native, schema-level object. It declares logical tables, the
relationships between them, facts, dimensions, and metrics — so every consumer reads the same
definitions. Let's inspect it, then query metrics directly with the `SEMANTIC_VIEW(...)` clause.

In [ ]:
# What metrics and dimensions does the view expose?
session.sql("SHOW SEMANTIC METRICS IN SMS_MARKETING_SV").show()
session.sql("SHOW SEMANTIC DIMENSIONS IN SMS_MARKETING_SV").show()

In [ ]:
# Query the view directly — this is exactly the SQL a BI tool or Analyst emits.
# Marquee insight: automated flows earn far more per send than one-time broadcasts.
session.sql("""
    SELECT * FROM SEMANTIC_VIEW(
        SMS_MARKETING_SV
        DIMENSIONS campaigns.campaign_type
        METRICS orders.attributed_revenue, revenue_per_send, ctr, send_count
    ) ORDER BY revenue_per_send DESC
""").show()

In [ ]:
# Derived metrics compose cleanly: consent + churn by region, from the SAME definitions.
session.sql("""
    SELECT * FROM SEMANTIC_VIEW(
        SMS_MARKETING_SV
        DIMENSIONS subscribers.region
        METRICS consent_rate, list_churn_rate, subscriber_count
    ) ORDER BY consent_rate DESC
""").show()

---
## Section 3: Three ways to build the same object

The view above was created programmatically by `setup.sql`. There are three on-ramps — all
producing the **identical** OSI-format object.

### Path A — Programmatic (DDL)
`CREATE SEMANTIC VIEW ... TABLES(...) RELATIONSHIPS(...) FACTS(...) DIMENSIONS(...) METRICS(...)`.
Lives in Git next to your dbt project; deploy via dbt or Terraform. Retrieve the exact DDL with
`GET_DDL`.

In [ ]:
# The full DDL of the deployed view (round-trips to source control)
print(session.sql("SELECT GET_DDL('SEMANTIC_VIEW', 'SMS_MARKETING_DEMO.CORE.SMS_MARKETING_SV')").collect()[0][0])

### Path B — CoCo-assisted, and generate from existing dbt models

Two flavors:

- **`/semantic-views` skill in Cortex Code** bootstraps or audits a view from your tables.
- **Generate from dbt** — if you already model metrics in dbt, promote them into a governed
  native object instead of starting over. A dbt `schema.yml` maps almost 1:1 to the view:

```yaml
# models/marts/schema.yml  (dbt) — the metric logic you already own
models:
  - name: fact_order
    columns:
      - name: revenue
        description: "Attributed order revenue (USD)"
    # dbt metrics / semantic model
    # metrics:
    #   - name: attributed_revenue
    #     expr: sum(revenue)
    #   - name: revenue_per_send
    #     expr: sum(revenue) / count(message_id)
```

Those `metrics` become the view's `METRICS`; dbt `relationships` tests become `RELATIONSHIPS`.
Zero-to-semantic-view in minutes, reusing logic that already exists. Then round-trip to YAML:

In [ ]:
# Export the view to YAML — commit this next to your dbt project (no lock-in)
print(session.sql("SELECT SYSTEM$READ_YAML_FROM_SEMANTIC_VIEW('SMS_MARKETING_DEMO.CORE.SMS_MARKETING_SV')").collect()[0][0])

### Path C — No-code (Snowsight Semantic View Generator wizard)

For analysts who prefer clicks over code:

1. Snowsight → **AI & ML → Cortex Analyst**.
2. **Create → Semantic View**; choose database `SMS_MARKETING_DEMO`, schema `CORE`.
3. Add the five tables; confirm the auto-detected joins (FKs → PKs).
4. Promote columns to **dimensions** (region, channel, campaign_type, theme, months) and
   define **metrics** (attributed_revenue, revenue_per_send, ctr, ...).
5. Add **synonyms** and **sample values** so Analyst understands business language.
6. Add a few **verified queries** from real questions, then **Save**.

The wizard writes the same object — round-trippable to YAML/DDL via the functions above.

---
## Section 4: Cortex Analyst — natural language, governed SQL

Cortex Analyst maps a marketer's plain-English question to the view's metrics, dimensions, and
synonyms (never to raw columns), and emits governed SQL against `SEMANTIC_VIEW(...)`. Below are
the five demo questions and the governed SQL each resolves to — run them here, then ask the same
questions in the Analyst UI (or the agent) and compare.

In [ ]:
# Q1: "Attributed revenue by campaign type"
session.sql("""
    SELECT * FROM SEMANTIC_VIEW(SMS_MARKETING_SV
        DIMENSIONS campaigns.campaign_type
        METRICS orders.attributed_revenue)
    ORDER BY attributed_revenue DESC
""").show()

# Q2: "Which region has the fastest opt-in growth?"
session.sql("""
    SELECT * FROM SEMANTIC_VIEW(SMS_MARKETING_SV
        DIMENSIONS subscribers.region
        METRICS subscribers.opt_in_growth)
    ORDER BY opt_in_growth DESC
""").show()

In [ ]:
# Q3: "Revenue per send — flows vs broadcasts"
session.sql("""
    SELECT * FROM SEMANTIC_VIEW(SMS_MARKETING_SV
        DIMENSIONS campaigns.campaign_type
        METRICS revenue_per_send)
    ORDER BY revenue_per_send DESC
""").show()

# Q4: "Click-through rate by campaign theme"
session.sql("""
    SELECT * FROM SEMANTIC_VIEW(SMS_MARKETING_SV
        DIMENSIONS campaigns.theme
        METRICS ctr)
    ORDER BY ctr DESC
""").show()

# Q5: "Subscriber LTV by region"
session.sql("""
    SELECT * FROM SEMANTIC_VIEW(SMS_MARKETING_SV
        DIMENSIONS subscribers.region
        METRICS subscriber_ltv)
    ORDER BY subscriber_ltv DESC
""").show()

> **Why it's accurate:** the relationships and metric expressions are declared once in the
> view. Analyst never guesses how `messages` joins to `orders`, and "list growth" / "opt-in
> growth" / "new subscribers" all resolve to the same metric via synonyms. Verified queries in
> the view anchor the most common questions.

---
## Section 5: Cortex Search — grounded, cited retrieval

Not everything is a number. Campaign briefs, copy, TCPA/consent rules, deliverability, a
segmentation playbook, support macros, a quarterly performance review, an attribution
whitepaper, and an incident postmortem live in documents. `SMS_DOCS_SEARCH` indexes them for
semantic, cited retrieval.

> **Real PDFs:** the 10 documents are real PDFs shipped in `lab/docs/`. `setup.sql` PUTs them to
> `@SMS_DOCS`, runs `ALTER STAGE ... REFRESH`, and parses each with
> `SNOWFLAKE.CORTEX.PARSE_DOCUMENT(@SMS_DOCS, RELATIVE_PATH, {'mode':'LAYOUT'})` into
> `SMS_DOC_CHUNKS` (with `doc_type` inferred from the filename) before building the service.

In [ ]:
import json

# Why do flows outperform broadcasts? Ask the document corpus.
resp = session.sql("""
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SMS_MARKETING_DEMO.CORE.SMS_DOCS_SEARCH',
        '{ "query": "why do flows outperform broadcasts",
           "columns": ["title","doc_type"], "limit": 3 }'
    )
""").collect()[0][0]

for r in json.loads(resp)["results"]:
    print(f"- {r['title']}  [{r['doc_type']}]")

In [ ]:
# Filter retrieval by doc_type — e.g. only the incident postmortem the agent will cite later
resp = session.sql("""
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SMS_MARKETING_DEMO.CORE.SMS_DOCS_SEARCH',
        '{ "query": "what caused the PNW flash sale throughput issue",
           "columns": ["title","doc_type","text"],
           "filter": { "@eq": { "doc_type": "postmortem" } },
           "limit": 2 }'
    )
""").collect()[0][0]

for r in json.loads(resp)["results"]:
    print(f"### {r['title']} [{r['doc_type']}]\n{r['text'][:300]}...\n")

---
## Section 5b: Call transcripts — ingestion patterns + a second Search service

The document corpus above is polished PDFs. Real "voice of the customer" data is messier: **call
transcripts** — multi-speaker, conversational, timestamped. That's the hardest corpus for keyword
search and exactly where Cortex Search's hybrid vector + keyword retrieval wins.

`setup.sql` ships 24 synthetic transcripts (10 support, 8 sales, 6 compliance) in
`lab/transcripts/`, lands them on `@CALL_TRANSCRIPTS_STAGE`, chunks them into `TRANSCRIPT_CHUNKS`,
and indexes them with a second service, **`CALL_TRANSCRIPTS_SEARCH`**. Along the way it shows
several **ingestion patterns** so you can narrate the trade-offs:

| Pattern | What | When |
|--------|------|------|
| **A — chunk directly off the stage** | Reassemble + chunk staged files in one query, nothing persisted | Exploration, or the stage *is* your source of record |
| **B — COPY INTO a raw table (ELT)** | Bulk-load lines → reassemble → chunk | Durable, re-queryable — the production path here |
| **C — Snowpipe / Snowpipe Streaming** | Auto-ingest new files, or append live rows | Continuous / real-time (narrated in `setup.sql`) |

In [ ]:
# What landed on the transcript stage, and how it flowed into the chunk table.
print("Files on @CALL_TRANSCRIPTS_STAGE:")
for r in session.sql("LIST @CALL_TRANSCRIPTS_STAGE").collect()[:5]:
    print("  ", r["name"])

counts = session.sql("""
    SELECT
        (SELECT COUNT(*) FROM CALL_MANIFEST)                     AS manifest_rows,
        (SELECT COUNT(*) FROM CALL_TRANSCRIPTS_RAW)              AS raw_transcripts,
        (SELECT COUNT(DISTINCT call_id) FROM TRANSCRIPT_CHUNKS)  AS chunked_calls,
        (SELECT COUNT(*) FROM TRANSCRIPT_CHUNKS)                 AS total_chunks
""").collect()[0]
print(f"\nmanifest={counts['MANIFEST_ROWS']}  raw={counts['RAW_TRANSCRIPTS']}  "
      f"chunked_calls={counts['CHUNKED_CALLS']}  chunks={counts['TOTAL_CHUNKS']}")

# call_type is derived from the CALL_ID prefix (CS=support, SD/SE=sales, CE=compliance)
session.sql("""
    SELECT call_type, COUNT(DISTINCT call_id) AS calls, COUNT(*) AS chunks
    FROM TRANSCRIPT_CHUNKS GROUP BY call_type ORDER BY calls DESC
""").show()

In [ ]:
# Pattern A — chunk DIRECTLY off the stage, nothing persisted. Same shape as the
# production table, but produced in a single query with no landing table.
pattern_a = session.sql("""
    SELECT
        REGEXP_SUBSTR(f.transcript_text, 'CALL_ID:\\\\s*(\\\\S+)', 1, 1, 'e', 1) AS call_id,
        c.index                                                                 AS chunk_index,
        LEFT(c.value::STRING, 90)                                               AS chunk_preview
    FROM (
        SELECT REGEXP_REPLACE(METADATA$FILENAME, '^.*/', '')                        AS file_name,
               LISTAGG($1, '\\n') WITHIN GROUP (ORDER BY METADATA$FILE_ROW_NUMBER)  AS transcript_text
        FROM @CALL_TRANSCRIPTS_STAGE (FILE_FORMAT => 'FF_TRANSCRIPT_LINES',
                                      PATTERN => '.*transcript_.*[.]txt')
        GROUP BY file_name
    ) f,
    LATERAL FLATTEN(input => SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
                                 f.transcript_text, 'none', 1200, 200)) c
    ORDER BY call_id, chunk_index
    LIMIT 5
""")
pattern_a.show()

# Pattern B produced the durable TRANSCRIPT_CHUNKS via COPY INTO CALL_TRANSCRIPTS_RAW_LINES,
# then LISTAGG reassembly. Both paths yield identical chunk counts:
print("Pattern B durable chunk count:",
      session.sql("SELECT COUNT(*) FROM TRANSCRIPT_CHUNKS").collect()[0][0])

In [ ]:
# One chunk row: the manifest join enriches every chunk with brand, call_date, and summary,
# which become filterable Search attributes.
session.sql("""
    SELECT chunk_id, call_id, call_type, brand, call_date, chunk_index,
           LEFT(chunk_text, 120) AS chunk_preview
    FROM TRANSCRIPT_CHUNKS
    WHERE brand = 'Harbor & Pine Home'
    ORDER BY call_id, chunk_index
    LIMIT 3
""").show()

### Search the transcripts — hybrid retrieval + attribute filters

`CALL_TRANSCRIPTS_SEARCH` indexes `chunk_text` with `call_type`, `brand`, and `call_date` as
filterable attributes. No vector store to manage, no embedding pipeline to maintain — the service
handles hybrid vector + keyword retrieval and stays fresh via `TARGET_LAG`.

In [ ]:
# Q: a deliverability / 10DLC throttling problem — messy phrasing, no exact keywords.
resp = session.sql("""
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SMS_MARKETING_DEMO.CORE.CALL_TRANSCRIPTS_SEARCH',
        '{ "query": "carrier filtering and 10DLC throttling causing a deliverability drop",
           "columns": ["call_id","brand","call_type","chunk_text"], "limit": 3 }'
    )
""").collect()[0][0]

for r in json.loads(resp)["results"]:
    print(f"### {r['brand']} [{r['call_type']}]  {r['call_id']}\n{r['chunk_text'][:280]}...\n")

In [ ]:
# Same service, scoped with an ATTRIBUTE FILTER: TCPA consent / STOP handling on COMPLIANCE calls
# only. Filtering happens at retrieval time — no re-indexing, RBAC inherited.
resp = session.sql("""
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SMS_MARKETING_DEMO.CORE.CALL_TRANSCRIPTS_SEARCH',
        '{ "query": "express written consent dispute and STOP opt-out handling",
           "columns": ["call_id","brand","call_type","chunk_text"],
           "filter": { "@eq": { "call_type": "compliance" } },
           "limit": 3 }'
    )
""").collect()[0][0]

for r in json.loads(resp)["results"]:
    print(f"### {r['brand']} [{r['call_type']}]  {r['call_id']}\n{r['chunk_text'][:280]}...\n")

### RAG closer — retrieve, then generate a cited answer

Retrieval feeds the LLM grounded context, killing hallucination on "what did the customer say?".
Pull the top chunks with `SEARCH_PREVIEW`, hand them to `AI_COMPLETE`, and ask for an answer that
cites the brand and `call_id` for every point.

In [ ]:
# Retrieve top transcript chunks for the question...
question = "what caused the attribution discrepancy between our platform and Shopify revenue"
resp = session.sql(f"""
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SMS_MARKETING_DEMO.CORE.CALL_TRANSCRIPTS_SEARCH',
        '{{ "query": "{question}",
           "columns": ["call_id","brand","chunk_text"], "limit": 5 }}'
    )
""").collect()[0][0]
hits = json.loads(resp)["results"]
context = "\n---\n".join(f"{h['brand']} [{h['call_id']}]: {h['chunk_text']}" for h in hits)

# ...then ground AI_COMPLETE on ONLY those excerpts, and ask for citations.
answer = session.sql("""
    SELECT SNOWFLAKE.CORTEX.AI_COMPLETE(
        'llama3.1-70b',
        CONCAT(
          'You are a support analyst. Using ONLY the call transcript excerpts below, explain what ',
          'caused the attribution discrepancy between our platform and Shopify, and cite the brand ',
          'and call_id for each point. If the excerpts do not answer it, say so.\n\nEXCERPTS:\n',
          ?
        )
    )
""", params=[context]).collect()[0][0]
print(answer)

---
## Section 6: Cortex Agent — blend structured + unstructured

`SMS_MARKETING_AGENT` has three tools: the semantic view via **Cortex Analyst** (governed KPIs),
**Marketing_Playbook_Search** (our policy/playbook PDFs), and **Call_Transcript_Search** (the
customer call transcripts from Section 5b). Model = auto. Ask a blended question and it routes: the
number to Analyst, the policy to Playbook Search, and *what the customer said* to Transcript
Search — all governed by the objects you already defined.

Inspect the agent's specification:

In [ ]:
session.sql("DESCRIBE AGENT SMS_MARKETING_AGENT").show()

### Chat with the agent

The best experience is interactive: **Snowsight → AI & ML → Agents → SMS_MARKETING_AGENT** (or via
Snowflake Intelligence). Ask the marquee blended questions:

> *"Attributed revenue for the Q3 flash sale came in below plan — what does the incident postmortem
> say caused the PNW throughput issue, and which campaign brief covered that send?"*

> *"Compare revenue per send for flows versus broadcasts, and explain from our playbook why flows
> perform the way they do."*

You can also drive it programmatically with `SNOWFLAKE.CORTEX.DATA_AGENT_RUN` or the REST endpoint
`POST /api/v2/databases/SMS_MARKETING_DEMO/schemas/CORE/agents/SMS_MARKETING_AGENT:run`. The cell below shows the
Analyst-tool half of the first question — the exact governed SQL the agent runs for the number:

In [ ]:
# The "number" half the agent gets from its Analyst tool: attributed revenue by campaign type
session.sql("""
    SELECT * FROM SEMANTIC_VIEW(SMS_MARKETING_SV
        DIMENSIONS campaigns.campaign_type
        METRICS orders.attributed_revenue, revenue_per_send)
    ORDER BY attributed_revenue DESC
""").show()

---
## Section 7: Programmatically build & optimize the agent

`setup.sql` already deployed an **optimized** `SMS_MARKETING_AGENT`. Here we show *how* it got that
way using the **/agent-optimization** workflow — build a deliberately weak **baseline** agent
programmatically, then apply the best practices that matter most.

**The single highest-leverage factor is tool descriptions.** Vague descriptions cascade into wrong
tool selection → irrelevant data → hallucinated answers. The optimization also:
- separates **orchestration** instructions (role, domain context, tool-selection rules, business
  rules) from **response** instructions (tone, formatting, units, citations);
- gives each tool a clear **name**, **coverage**, and explicit **when-to-use / when-NOT-to-use**
  boundaries so the agent never confuses metric questions with policy questions;
- seeds **sample questions** so users (and the Snowsight playground) start in the right place.

> **Safety:** never optimize a **production** agent in place. Clone it first
> (`CREATE AGENT <db>.<schema>.<clone> CLONE <prod_agent>`, or redeploy its spec under a new name),
> optimize the clone, evaluate, then promote. The agents here are demo objects, so we build them
> directly. A versioned record of this optimization lives in `agent_optimization/`.

In [ ]:
# BEFORE — a deliberately weak baseline: vague tool descriptions, no orchestration,
# no response guidance, no sample questions. This is what "just wire up the tools" looks like.
session.sql("""
CREATE OR REPLACE AGENT SMS_MARKETING_AGENT_BASELINE
  COMMENT = 'Deliberately un-optimized baseline for the agent-optimization demo'
  PROFILE = '{"display_name": "SMS Agent (baseline)", "color": "gray"}'
  FROM SPECIFICATION
$$
models:
  orchestration: auto
instructions:
  response: "Answer the question."
tools:
  - tool_spec:
      type: "cortex_analyst_text_to_sql"
      name: "analyst"
      description: "Marketing data."
  - tool_spec:
      type: "cortex_search"
      name: "search"
      description: "Documents."
tool_resources:
  analyst:
    semantic_view: "SMS_MARKETING_DEMO.CORE.SMS_MARKETING_SV"
    execution_environment:
      type: warehouse
      warehouse: "SMS_MARKETING_WH"
  search:
    name: "SMS_MARKETING_DEMO.CORE.SMS_DOCS_SEARCH"
    max_results: "5"
    title_column: "title"
    id_column: "doc_id"
$$
""").show()

### AFTER — apply the best practices

| Dimension | Baseline | Optimized |
|---|---|---|
| Tool names | `analyst`, `search` | `Marketing_KPI_Analyst`, `Marketing_Playbook_Search` |
| Tool descriptions | "Marketing data." / "Documents." | coverage + metrics/dimensions + when-to-use + **when NOT to use** |
| Orchestration | none | role, domain context, tool-selection rules, business rules |
| Response | "Answer the question." | units (USD / %), tables for >3 rows, cite by title |
| Sample questions | none | 4 marquee questions (incl. a blended one) |
| Chart tool | none | `data_to_chart` added |

The optimized spec below is the same one `setup.sql` ships. Deploying it programmatically here shows
the full build path.

In [ ]:
# AFTER — deploy the optimized agent programmatically (identical to setup.sql's spec).
session.sql("""
CREATE OR REPLACE AGENT SMS_MARKETING_AGENT
  COMMENT = 'AI-BI assistant for the SMS/MMS marketing platform (optimized per agent best practices)'
  PROFILE = '{"display_name": "SMS Marketing Analyst", "color": "blue"}'
  FROM SPECIFICATION
$$
models:
  orchestration: auto

instructions:
  response: |
    - Lead with the direct answer, then supporting detail. Be concise.
    - Report currency as USD (e.g. $59.29) and rates as percentages (e.g. 13.5%).
    - Use a table for multi-row results (more than 3 rows); state single values inline.
    - When you use a document, cite it by its title.
    - If a metric is unavailable or out of scope, say so and suggest the closest available metric.
  orchestration: |
    Role: You are the SMS Marketing Analyst for an SMS/MMS marketing platform used by e-commerce brands. Users are marketing managers and analysts.
    Domain context:
    - Campaign types: "broadcast" (one-time blast) and "flow" (automated, triggered: welcome, abandoned-cart, winback). Flows fire at high intent and usually earn far more revenue per send than broadcasts.
    - consent_status: opted_in (marketable), opted_out (churned), pending (unconfirmed). Consent rate = opted_in / total; list churn rate = opted_out / total.
    - Revenue is last-touch attributed from store orders to the send that drove them. Regions are NE, SE, MW, SW, W.
    Tool selection:
    - Use Marketing_KPI_Analyst for numbers, rates, trends, rankings, or comparisons of metrics sliced by region, channel, campaign type, theme, plan tier, industry, or month.
    - Use Marketing_Playbook_Search for how / why / policy / copy questions answered by our own documents.
    - Use Call_Transcript_Search for the voice of the customer: what was said on a specific support, sales, or compliance call (objections, complaints, root-cause discussions). Scope with call_type/brand/call_date filters when named.
    - For blended "what happened and why" questions, get the number from Marketing_KPI_Analyst, the policy/brief from Marketing_Playbook_Search, and what the customer said from Call_Transcript_Search, then combine.
    - Playbook_Search = OUR policy/playbook docs; Call_Transcript_Search = transcripts of CALLS with customers.
    Business rules:
    - Do not conflate historical aggregates with predictions; this agent does not forecast.
    - If the time range is ambiguous, default to the last 12 months and say so.
    - Only reference metrics defined in the semantic view; never invent a metric.
  sample_questions:
    - question: "What is attributed revenue by campaign type, and how does revenue per send compare between flows and broadcasts?"
    - question: "Which region has the fastest opt-in growth, and what is its consent rate and list churn?"
    - question: "Attributed revenue for the Q3 flash sale came in below plan — what does the incident postmortem say caused the PNW throughput issue, and which campaign brief covered that send?"
    - question: "What does our playbook say about TCPA consent and quiet hours before a broadcast?"
    - question: "Trace the brand that had a 10DLC campaign suspension across its support and compliance calls — what did the customer say, and what was the remediation?"

tools:
  - tool_spec:
      type: "cortex_analyst_text_to_sql"
      name: "Marketing_KPI_Analyst"
      description: |
        Governed marketing KPIs from the SMS_MARKETING_SV semantic view (text-to-SQL over a star schema).
        Data coverage: brands, subscribers, campaigns, message sends, and last-touch attributed orders for ~18 months.
        Metrics: attributed_revenue, revenue_per_send, ctr, opt_in_growth, subscriber_ltv, list_churn_rate, consent_rate, plus counts.
        Dimensions: region (NE/SE/MW/SW/W), channel (SMS/MMS), campaign_type (broadcast/flow), theme, plan_tier, industry, opt_in_month, send_month, order_month.
        When to use: questions about numbers, rates, trends, rankings, or comparisons of the metrics above.
        When NOT to use: how/why/policy questions, message copy, or TCPA/deliverability guidance (use Marketing_Playbook_Search); forecasting (not supported).
        Query tips: name the metric and slice explicitly; use exact region codes; give a month range for trends.
  - tool_spec:
      type: "cortex_search"
      name: "Marketing_Playbook_Search"
      description: |
        Semantic search over the marketing playbook and policy corpus (real PDFs): campaign briefs, SMS/MMS copy library, TCPA/consent guidelines, deliverability best practices, segmentation playbook, support macros, quarterly performance review, attribution whitepaper, and incident postmortem.
        When to use: "how", "why", "what's the policy", "what should the copy say", "which brief", "what happened in the incident", or "what did the quarterly review say" questions.
        When NOT to use: numeric/metric questions (use Marketing_KPI_Analyst).
        Filterable attributes: doc_type (campaign_brief, copy_library, tcpa_consent, deliverability, segmentation, support_macro, performance_review, attribution, postmortem) and region (currently ALL). Cite results by title.
  - tool_spec:
      type: "cortex_search"
      name: "Call_Transcript_Search"
      description: |
        Semantic search over 24 synthetic customer CALL TRANSCRIPTS (10 support, 8 sales, 6 compliance), chunked to preserve dialogue turns.
        When to use: "what did the customer say", "what happened on the call", objections, complaints, deliverability/10DLC/attribution root-cause, consent/STOP/TCPA disputes, or tracing one brand's issue across calls.
        When NOT to use: aggregate numbers/rates (Marketing_KPI_Analyst); our own policy/copy/playbook (Marketing_Playbook_Search).
        Filterable attributes: call_type (support, sales, compliance), brand, call_date. Cite by brand and call_id.
  - tool_spec:
      type: "data_to_chart"
      name: "data_to_chart"
      description: "Generates a chart from tabular results when a visualization helps a comparison, trend, or ranking."

tool_resources:
  Marketing_KPI_Analyst:
    semantic_view: "SMS_MARKETING_DEMO.CORE.SMS_MARKETING_SV"
    execution_environment:
      type: warehouse
      warehouse: "SMS_MARKETING_WH"
  Marketing_Playbook_Search:
    name: "SMS_MARKETING_DEMO.CORE.SMS_DOCS_SEARCH"
    max_results: "5"
    title_column: "title"
    id_column: "doc_id"
  Call_Transcript_Search:
    name: "SMS_MARKETING_DEMO.CORE.CALL_TRANSCRIPTS_SEARCH"
    max_results: "6"
    title_column: "call_id"
    id_column: "chunk_id"
$$
""").show()

In [ ]:
# Confirm both agents exist, then A/B them in Snowsight → AI & ML → Agents:
# ask the same blended question of SMS_MARKETING_AGENT_BASELINE vs SMS_MARKETING_AGENT and
# compare tool routing and answer quality.
session.sql("SHOW AGENTS LIKE 'SMS_MARKETING_AGENT%'").show()

### Take it to production with the full /agent-optimization loop

This section showed the **build + best-practices** half. The full workflow adds systematic evaluation:

1. **System of record** — a versioned workspace per agent (`agent_optimization/` in this module):
   baseline spec, optimized spec, and a running `optimization_log.md`.
2. **Eval dataset** — 15-20 curated questions with expected answers
   (`agent_optimization/eval_questions.md`), ~25% testing tool routing.
3. **Baseline eval → failure analysis** — run the eval, group failures (wrong tool, formatting, or
   SQL issues that actually belong in the semantic view).
4. **Instruction fixes → re-eval** — target the failure patterns and measure the lift.
5. **Overfitting check → generalize** — remove eval-specific hardcoding so it holds in production.
6. **Clone-first for production agents**, then promote the winning version.

In Cortex Code, run **/agent-optimization** and choose OPTIMIZE to drive this end to end with the
evaluation scripts (`run_evaluation.py`, `create_or_alter_agent.py`, `get_agent_config.py`).

---
## Section 8: Positioning recap

1. **Define once** — the same `attributed_revenue` / `revenue_per_send` answered every question
   above, whether from raw SQL, Analyst, or the Agent. Define it in the platform, not in five tools.
2. **Layered, not versus** — dbt is the code-first system of record; the Semantic View is the
   native governed metric layer; BI tools (Omni, Tableau, Excel) are the render layer.
3. **Governed by default** — RBAC, row-access policies, masking, tagging, and lineage on the base
   tables are inherited by every consumer; nothing is bolted on per tool.
4. **Open & portable** — you exported the view to YAML in Section 3; commit it to Git next to dbt.
5. **AI-native** — the view is the accuracy foundation that kills text-to-SQL hallucination for
   Analyst and Agents.

---
## Summary

You built and exercised a complete AI-BI stack on one governed object:

- A **semantic view** defining 7 KPIs once (attributed revenue, revenue per send, CTR, opt-in
  growth, subscriber LTV, list churn, consent rate).
- **Three creation paths** (DDL, YAML/dbt, no-code wizard) — all the same object, exportable to Git.
- **Cortex Analyst** returning governed SQL, **Cortex Search** returning cited documents, and a
  **Cortex Agent** blending both — built programmatically and **optimized** with the
  /agent-optimization best practices (baseline → optimized).

**Tear down** with `lab/cleanup.sql` when you're done, or re-run `lab/setup.sql` to reset.